<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 15: Ai Etik

**YAPAY ZEKA MÜHENDİSLİĞİ** · Modül 15 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta15/hafta15_ai_etik.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta15/hafta15_ai_etik.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 15 - Yapay Zeka Etiği

Bu defterde yapay zeka sistemlerindeki etik sorunları, önyargı (bias) problemlerini ve adalet (fairness) kavramlarını inceleyeceğiz.

## Öğrenme Hedefleri
- AI önyargısı (bias) kavramını anlama
- Önyargılı veri seti oluşturma ve etkilerini gözlemleme
- Adalet metrikleri: Demografik parite, eşitlenmiş oranlar
- Önyargı azaltma stratejileri
- AB Yapay Zeka Yasası (EU AI Act) özeti
- Etik tartışma soruları

## 1. Yapay Zekada Önyargı (AI Bias) Nedir?

AI önyargısı, yapay zeka sistemlerinin belirli grupları sistematik olarak kayırması veya dezavantajlı duruma düşürmesidir.

### Önyargı Kaynakları

```
┌────────────────┐     ┌─────────────────┐     ┌────────────────┐
│  VERİ ÖNYARGISI │     │ ALGORİTMA        │     │ SONUÇ ÖNYARGISI│
│                │     │ ÖNYARGISI         │     │                │
│ - Eksik temsil │ ──→ │ - Yanlış ağırlık │ ──→ │ - Ayrımcı      │
│ - Tarihsel     │     │ - Uygun olmayan   │     │   kararlar     │
│   ayrımcılık   │     │   metrik seçimi   │     │ - Eşitsiz      │
│ - Örnekleme    │     │ - Aşırı öğrenme   │     │   fırsatlar    │
│   hatası       │     │                   │     │                │
└────────────────┘     └─────────────────┘     └────────────────┘
```

### Gerçek Dünya Örnekleri

| Vaka | Sorun | Etki |
|------|-------|------|
| **Amazon İşe Alım AI** (2018) | Kadın adayları sistematik olarak düşük puanladı | Proje iptal edildi |
| **COMPAS Suç Tahmini** (2016) | Siyahi bireylere yanlış yüksek risk verdi | Adalet sisteminde eşitsizlik |
| **Yüz Tanıma** (çeşitli) | Koyu tenli yüzlerde düşük doğruluk | Hatalı kimlik tespiti |
| **Google Translate** | Cinsiyet stereotipleri (doktor=erkek, hemşire=kadın) | Toplumsal önyargıları pekiştirme |
| **Kredi Puanlama** | Posta kodu bazlı ayrımcılık | Dezavantajlı bölgelere düşük kredi |

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Kütüphaneler hazır!")

## 2. Önyargılı Veri Seti Oluşturma: Kredi Onayı

Kredi başvurusu senaryosu: Cinsiyet ve bölge bazında gizli önyargı içeren bir veri seti oluşturacağız.

In [ ]:
n = 2000  # Toplam başvuru sayısı

# Özellikler
cinsiyet = np.random.choice(['Erkek', 'Kadın'], size=n, p=[0.6, 0.4])
yas = np.random.randint(22, 65, size=n)
gelir = np.random.normal(5000, 2000, size=n).clip(1500, 20000).astype(int)
egitim = np.random.choice(['Lise', 'Üniversite', 'Yüksek Lisans'], size=n, p=[0.3, 0.5, 0.2])
kredi_gecmisi = np.random.randint(300, 850, size=n)  # Kredi puanı
bolge = np.random.choice(['Merkez', 'Kırsal', 'Kenar Mahalle'], size=n, p=[0.4, 0.3, 0.3])

# ÖNYARGILI onay kararı oluşturma
onay_olasiligi = np.zeros(n)

# Temel faktörler (adil)
onay_olasiligi += (gelir - 1500) / 18500 * 0.3  # Gelir etkisi
onay_olasiligi += (kredi_gecmisi - 300) / 550 * 0.3  # Kredi geçmişi etkisi

# Eğitim etkisi (adil)
egitim_puan = np.where(egitim == 'Yüksek Lisans', 0.15,
              np.where(egitim == 'Üniversite', 0.10, 0.05))
onay_olasiligi += egitim_puan

# ÖNYARGI 1: Cinsiyet önyargısı - kadınlara düşük onay
cinsiyet_bias = np.where(cinsiyet == 'Kadın', -0.15, 0.05)
onay_olasiligi += cinsiyet_bias

# ÖNYARGI 2: Bölge önyargısı - kenar mahallelere düşük onay
bolge_bias = np.where(bolge == 'Kenar Mahalle', -0.20,
             np.where(bolge == 'Kırsal', -0.05, 0.10))
onay_olasiligi += bolge_bias

# Gürültü ekle ve ikili sonuca çevir
onay_olasiligi += np.random.normal(0, 0.1, size=n)
onay_olasiligi = np.clip(onay_olasiligi, 0, 1)
onay = (onay_olasiligi > 0.45).astype(int)

# DataFrame oluştur
df = pd.DataFrame({
    'cinsiyet': cinsiyet,
    'yas': yas,
    'gelir': gelir,
    'egitim': egitim,
    'kredi_puani': kredi_gecmisi,
    'bolge': bolge,
    'onay': onay  # 1=Onaylandı, 0=Reddedildi
})

print(f"Veri seti boyutu: {df.shape}")
print(f"\nOnay oranı: {df['onay'].mean():.2%}")
df.head(10)

## 3. Önyargıyı Görselleştirme

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cinsiyet bazında onay oranı
cinsiyet_onay = df.groupby('cinsiyet')['onay'].mean()
colors = ['#e74c3c' if v < cinsiyet_onay.mean() else '#2ecc71' for v in cinsiyet_onay.values]
cinsiyet_onay.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Cinsiyet Bazında Kredi Onay Oranı', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Onay Oranı')
axes[0].set_xlabel('')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(cinsiyet_onay.values):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold', fontsize=13)

# Bölge bazında onay oranı
bolge_onay = df.groupby('bolge')['onay'].mean()
colors2 = ['#e74c3c' if v < bolge_onay.mean() else '#2ecc71' for v in bolge_onay.values]
bolge_onay.plot(kind='bar', ax=axes[1], color=colors2, edgecolor='black')
axes[1].set_title('Bölge Bazında Kredi Onay Oranı', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Onay Oranı')
axes[1].set_xlabel('')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(bolge_onay.values):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

print("Fark (Cinsiyet):", f"{cinsiyet_onay['Erkek'] - cinsiyet_onay['Kadın']:.1%} Erkek lehine")

## 4. Önyargılı Veriyle Model Eğitimi

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Özellikleri hazırla
df_encoded = pd.get_dummies(df, columns=['cinsiyet', 'egitim', 'bolge'], drop_first=False)

X = df_encoded.drop('onay', axis=1)
y = df_encoded['onay']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Model eğitimi
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

y_pred = model_rf.predict(X_test)

print("Model Performansı (Genel):")
print(f"Doğruluk (Accuracy): {accuracy_score(y_test, y_pred):.2%}")
print()
print(classification_report(y_test, y_pred, target_names=['Reddedildi', 'Onaylandı']))

## 5. Adaletsiz Tahminleri Gösterme

Model genel olarak iyi performans gösterse de, alt gruplara baktığımızda önyargıyı görebiliriz.

In [ ]:
# Test verisine tahminleri ekle
test_sonuc = X_test.copy()
test_sonuc['gercek'] = y_test.values
test_sonuc['tahmin'] = y_pred

# Cinsiyet bazında performans
print("=" * 50)
print("CİNSİYET BAZINDA TAHMİN SONUÇLARI")
print("=" * 50)

for cinsiyet_col in ['cinsiyet_Erkek', 'cinsiyet_Kadın']:
    grup = test_sonuc[test_sonuc[cinsiyet_col] == 1]
    cinsiyet_adi = cinsiyet_col.split('_')[1]
    
    dogruluk = accuracy_score(grup['gercek'], grup['tahmin'])
    onay_orani = grup['tahmin'].mean()
    gercek_onay = grup['gercek'].mean()
    
    print(f"\n{cinsiyet_adi}:")
    print(f"  Kişi sayısı: {len(grup)}")
    print(f"  Doğruluk: {dogruluk:.2%}")
    print(f"  Tahmin edilen onay oranı: {onay_orani:.2%}")
    print(f"  Gerçek onay oranı: {gercek_onay:.2%}")

### Senaryolu test: Aynı profildeki erkek ve kadın başvurusu

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Senaryolu test: Aynı profildeki erkek ve kadın başvurusu
print("\n" + "=" * 60)
print("SENARYO: Aynı profil, farklı cinsiyet")
print("=" * 60)
print("Profil: 35 yaş, 7000 TL gelir, Üniversite, 650 kredi puanı, Merkez")
print()

# Boş DataFrame oluştur (eğitim verisindeki sütunlarla aynı)
ornek_sutunlar = X_train.columns.tolist()

def profil_olustur(cinsiyet_erkek=1, cinsiyet_kadin=0):
    profil = pd.DataFrame(0, index=[0], columns=ornek_sutunlar)
    profil['yas'] = 35
    profil['gelir'] = 7000
    profil['kredi_puani'] = 650
    profil['cinsiyet_Erkek'] = cinsiyet_erkek
    profil['cinsiyet_Kadın'] = cinsiyet_kadin
    profil['egitim_Üniversite'] = 1
    profil['bolge_Merkez'] = 1
    return profil

erkek_profil = profil_olustur(cinsiyet_erkek=1, cinsiyet_kadin=0)
kadin_profil = profil_olustur(cinsiyet_erkek=0, cinsiyet_kadin=1)

erkek_tahmin = model_rf.predict(erkek_profil)[0]
kadin_tahmin = model_rf.predict(kadin_profil)[0]

erkek_olasilik = model_rf.predict_proba(erkek_profil)[0][1]
kadin_olasilik = model_rf.predict_proba(kadin_profil)[0][1]

print(f"Erkek başvurusu: {'ONAYLI' if erkek_tahmin else 'REDDEDİLDİ'} (Olasılık: {erkek_olasilik:.2%})")
print(f"Kadın başvurusu: {'ONAYLI' if kadin_tahmin else 'REDDEDİLDİ'} (Olasılık: {kadin_olasilik:.2%})")
print(f"\nFark: {erkek_olasilik - kadin_olasilik:.2%} erkek lehine")

if erkek_tahmin != kadin_tahmin:
    print("\n⚠️ ÖNYARGI TESPİTİ: Aynı profil, farklı sonuç!")

## 6. Adalet Metrikleri

### Demografik Parite (Demographic Parity)
Tüm grupların aynı onay oranına sahip olması gerekir.

$$P(\hat{Y}=1 | A=a) = P(\hat{Y}=1 | A=b)$$

### Eşitlenmiş Oranlar (Equalized Odds)
Tüm gruplar için doğru pozitif oranı (TPR) ve yanlış pozitif oranı (FPR) aynı olmalıdır.

$$P(\hat{Y}=1 | Y=y, A=a) = P(\hat{Y}=1 | Y=y, A=b) \quad \forall y \in \{0,1\}$$

In [ ]:
def adalet_metrikleri_hesapla(test_df, gercek_col, tahmin_col, grup_col):
    """Adalet metriklerini hesapla."""
    sonuclar = {}
    
    for grup_deger in test_df[grup_col].unique():
        if test_df[grup_col].dtype == 'object':
            grup = test_df[test_df[grup_col] == grup_deger]
        else:
            grup = test_df[test_df[grup_col] == grup_deger]
        
        if len(grup) == 0:
            continue
        
        gercek = grup[gercek_col]
        tahmin = grup[tahmin_col]
        
        # Demografik Parite: P(Y_hat=1 | A=a)
        onay_orani = tahmin.mean()
        
        # TPR (True Positive Rate / Recall)
        gercek_pozitif = gercek[gercek == 1]
        tahmin_pozitif = tahmin[gercek == 1]
        tpr = tahmin_pozitif.mean() if len(gercek_pozitif) > 0 else 0
        
        # FPR (False Positive Rate)
        gercek_negatif = gercek[gercek == 0]
        tahmin_negatif_grp = tahmin[gercek == 0]
        fpr = tahmin_negatif_grp.mean() if len(gercek_negatif) > 0 else 0
        
        sonuclar[grup_deger] = {
            'onay_orani': onay_orani,
            'tpr': tpr,
            'fpr': fpr,
            'sayi': len(grup)
        }
    
    return sonuclar

# Cinsiyet için adalet metrikleri
# Orijinal cinsiyet bilgisini geri ekle
test_sonuc['cinsiyet'] = np.where(test_sonuc['cinsiyet_Erkek'] == 1, 'Erkek', 'Kadın')

metrikler = adalet_metrikleri_hesapla(test_sonuc, 'gercek', 'tahmin', 'cinsiyet')

print("ADALET METRİKLERİ (Cinsiyet)")
print("=" * 60)
print(f"{'Grup':<15} {'Onay Oranı':<15} {'TPR':<10} {'FPR':<10} {'Sayı':<10}")
print("-" * 60)

for grup, degerler in metrikler.items():
    print(f"{grup:<15} {degerler['onay_orani']:<15.2%} {degerler['tpr']:<10.2%} {degerler['fpr']:<10.2%} {degerler['sayi']:<10}")

# Demografik parite farkı
gruplar = list(metrikler.keys())
if len(gruplar) == 2:
    dp_fark = abs(metrikler[gruplar[0]]['onay_orani'] - metrikler[gruplar[1]]['onay_orani'])
    print(f"\nDemografik Parite Farkı: {dp_fark:.2%}")
    print(f"Eşik: <5% adil, 5-10% şüpheli, >10% önyargılı")
    if dp_fark > 0.10:
        print("SONUÇ: Model ÖNYARGILI görünüyor!")
    elif dp_fark > 0.05:
        print("SONUÇ: Model ŞÜPHELİ, incelenmeli.")
    else:
        print("SONUÇ: Model görece ADİL görünüyor.")

## 7. Önyargı Azaltma Stratejileri

### Üç Ana Yaklaşım

| Aşama | Strateji | Açıklama |
|-------|----------|----------|
| **Veri Öncesi (Pre-processing)** | Veriyi düzelt | Dengesiz grupları dengele, hassas özellikleri kaldır |
| **Eğitim Sırası (In-processing)** | Algoritmayı düzelt | Adalet kısıtı ekle, düzenlileştirme uygula |
| **Eğitim Sonrası (Post-processing)** | Çıktıyı düzelt | Eşik ayarlama, kalibrasyon |

### Basit Uygulama: Hassas Özelliği Kaldırma

In [ ]:
# Strateji 1: Cinsiyet ve bölge özelliklerini kaldır
hassas_sutunlar = [col for col in X_train.columns if 'cinsiyet' in col or 'bolge' in col]
print(f"Kaldırılan hassas özellikler: {hassas_sutunlar}")

X_train_adil = X_train.drop(columns=hassas_sutunlar)
X_test_adil = X_test.drop(columns=hassas_sutunlar)

# Adil model eğitimi
model_adil = RandomForestClassifier(n_estimators=100, random_state=42)
model_adil.fit(X_train_adil, y_train)

y_pred_adil = model_adil.predict(X_test_adil)

print(f"\nÖnyargılı Model Doğruluğu: {accuracy_score(y_test, y_pred):.2%}")
print(f"Adil Model Doğruluğu: {accuracy_score(y_test, y_pred_adil):.2%}")

# Adil modelin metriklerini kontrol et
test_sonuc_adil = test_sonuc.copy()
test_sonuc_adil['tahmin_adil'] = y_pred_adil

metrikler_adil = adalet_metrikleri_hesapla(test_sonuc_adil, 'gercek', 'tahmin_adil', 'cinsiyet')

print("\nADİL MODEL METRİKLERİ")
print("=" * 50)
for grup, degerler in metrikler_adil.items():
    print(f"{grup}: Onay Oranı={degerler['onay_orani']:.2%}, TPR={degerler['tpr']:.2%}")

gruplar = list(metrikler_adil.keys())
if len(gruplar) == 2:
    dp_fark_adil = abs(metrikler_adil[gruplar[0]]['onay_orani'] - metrikler_adil[gruplar[1]]['onay_orani'])
    print(f"\nDemografik Parite Farkı (Adil): {dp_fark_adil:.2%}")
    print(f"Demografik Parite Farkı (Önyargılı): {dp_fark:.2%}")
    print(f"İyileşme: {dp_fark - dp_fark_adil:.2%}")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Karşılaştırmalı görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Önyargılı model
grup_adi = list(metrikler.keys())
onay_oranlari = [metrikler[g]['onay_orani'] for g in grup_adi]
axes[0].bar(grup_adi, onay_oranlari, color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0].set_title('Önyargılı Model', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Onay Oranı')
axes[0].set_ylim(0, 1)
axes[0].axhline(y=np.mean(onay_oranlari), color='gray', linestyle='--', label='Ortalama')
for i, v in enumerate(onay_oranlari):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[0].legend()

# Adil model
onay_oranlari_adil = [metrikler_adil[g]['onay_orani'] for g in grup_adi]
axes[1].bar(grup_adi, onay_oranlari_adil, color=['#3498db', '#e74c3c'], edgecolor='black')
axes[1].set_title('Adil Model (Hassas Özellikler Kaldırıldı)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Onay Oranı')
axes[1].set_ylim(0, 1)
axes[1].axhline(y=np.mean(onay_oranlari_adil), color='gray', linestyle='--', label='Ortalama')
for i, v in enumerate(onay_oranlari_adil):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. AB Yapay Zeka Yasası (EU AI Act)

Avrupa Birliği, 2024 yılında dünyanın ilk kapsamlı yapay zeka düzenlemesini kabul etmiştir.

### Risk Temelli Yaklaşım

| Risk Seviyesi | Tanım | Örnekler | Gereksinimler |
|---------------|-------|----------|---------------|
| **Kabul Edilemez Risk** | Yasaklanan AI uygulamaları | Sosyal puanlama, gerçek zamanlı toplu biyometrik gözetleme, manipülatif AI | Tamamen YASAK |
| **Yüksek Risk** | Temel haklara etki eden AI | Kredi puanlama, işe alım, adalet sistemi, eğitim, sağlık | Sıkı kurallar: şeffaflık, veri kalitesi, insan gözetimi |
| **Sınırlı Risk** | Etkileşim gerektiren AI | Chatbot'lar, deepfake üretimi, duygu tanıma | Şeffaflık yükümlülüğü (AI olduğunu bildirme) |
| **Düşük/Minimal Risk** | Çoğu AI uygulaması | Spam filtreleri, oyunlar, öneri sistemleri | Düzenleme YOK (gönüllü kurallar) |

### Yüksek Riskli AI Sistemleri İçin Gereksinimler
1. **Risk Yönetim Sistemi**: Sürekli risk değerlendirmesi
2. **Veri Yönetişimi**: Eğitim verisinin kalitesi, temsili ve önyargı kontrolü
3. **Teknik Dokümantasyon**: Sistemin çalışma prensibi ve sınırları
4. **Kayıt Tutma**: Otomatik log kaydı
5. **Şeffaflık**: Kullanıcılara bilgilendirme
6. **İnsan Gözetimi**: AI kararlarını denetleyen insan
7. **Doğruluk ve Güvenilirlik**: Performans standartları
8. **Siber Güvenlik**: Manipülasyona karşı koruma

### Cezalar
- Yasaklanan uygulamalar: **35 milyon Euro** veya global cironun **%7**'si
- Yüksek riskli ihlaller: **15 milyon Euro** veya global cironun **%3**'ü
- Yanlış bilgi verme: **7.5 milyon Euro** veya global cironun **%1.5**'i

### Türkiye İçin Durum
- Türkiye henüz AB üyesi değil ancak AB ile ticaret yapan şirketler EU AI Act'e uymak zorunda
- Türkiye'nin Ulusal Yapay Zeka Stratejisi (2021-2025) etik kuralları öneriyor ancak henüz yasal düzenleme yok
- TÜBİTAK ve ilgili kurumlar düzenleme çalışmalarını sürdürüyor

## 9. Tartışma Soruları

Aşağıdaki soruları grup olarak tartışın:

### Temel Sorular
1. **Önyargı kaçınılmaz mı?** Toplumda var olan önyargılar AI'ya yansıyorsa, AI bu önyargıları azaltabilir mi yoksa sadece pekiştirir mi?

2. **Adalet-performans dengesi**: Modelin genel doğruluğunu düşürerek adaleti artırmak etik midir? Bu dengeyi kim belirlemeli?

3. **Şeffaflık vs gizlilik**: AI sistemlerinin kararlarını açıklaması gerekli mi? Ticari sır ile şeffaflık arasındaki denge nasıl kurulmalı?

### Senaryo Tartışmaları

4. **Senaryo - İşe Alım**: Bir şirket AI tabanlı CV eleme sistemi kullanıyor. Sistem, kadın adayları %15 daha az oranda mülakata çağırıyor. Şirket ne yapmalı?
   - a) Sistemi kapatmalı
   - b) Cinsiyeti özelliklerden çıkarmalı
   - c) Sonuçları el ile dengelemeli
   - d) Başka bir çözüm?

5. **Senaryo - Sağlık**: Bir AI sistemi hastaların tedavi önceliğini belirliyor. Yaşlı hastalara düşük öncelik veriyor çünkü eğitim verisinde yaşlı hastalar daha az tedavi görmüş. Ne yapılmalı?

6. **Senaryo - Eğitim**: Bir AI sistemi öğrencilerin başarısını tahmin ediyor ve düşük puanlı öğrencilere daha az kaynak ayrılıyor. Bu self-fulfilling prophecy (kendini gerçekleştiren kehanet) yaratır mı?

### İleri Düzey Sorular

7. **Sorumlu kim?** AI kaynaklı bir hata sonucu birisi zarar görürse (yanlış teşhis, yanlış kredi reddi), sorumlu kim olmalı?
   - Geliştiriciler? Şirket? Kullanıcı? AI'ın kendisi?

8. **Küresel standart mümkün mü?** AB, ABD ve Çin'in AI etik yaklaşımları farklı. Küresel bir standart gerekli mi ve mümkün mü?

9. **Türkiye özelinde**: Türkiye'de AI etiği için hangi düzenlemeler yapılmalı? Eğitimde AI kullanımının sınırları ne olmalı?

## Özet

| Konu | Öğrenilen |
|------|----------|
| **AI Önyargısı** | Veri, algoritma ve sonuç önyargısı türleri |
| **Önyargılı Veri** | Cinsiyet ve bölge bazlı gizli önyargı etkisi |
| **Adaletsiz Tahminler** | Aynı profil farklı sonuç problemi |
| **Adalet Metrikleri** | Demografik parite, eşitlenmiş oranlar |
| **Azaltma Stratejileri** | Hassas özellik kaldırma, veri dengeleme |
| **EU AI Act** | Risk temelli düzenleme, yükümlülükler, cezalar |

### Alıştırma
1. Veri setine yaş bazlı önyargı ekleyin ve etkilerini analiz edin.
2. Farklı bir önyargı azaltma stratejisi uygulayın (veri dengeleme, eşik ayarlama).
3. Bir gerçek dünya AI sistemi seçin ve etik değerlendirme raporu yazın.

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://akademikyz.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>